## Cell 1 — Imports & Logging

In [0]:
import logging
import random
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_feature_table")

print("✓ Imports loaded")

## Cell 2 — Configuration

In [0]:
# ── Input path ────────────────────────────────────────────────
# We read from the neighborhood_summary serving table built in
# nb_Neighborhood_Summary.py — it already has all metrics joined
# and Census values cleaned. No need to re-read raw data.
SUMMARY_PATH = "/Volumes/workspace/default/serving/neighborhood_summary"

# ── Output path ───────────────────────────────────────────────
# Feature table lands in the serving volume alongside the
# analytical tables. Clearly named so it's obvious what it's for.
FEATURE_TABLE_PATH = "/Volumes/workspace/default/serving/ml_feature_table"

# ── Train/test split configuration ───────────────────────────
# 80% of neighborhoods go into training, 20% into test.
# With 63 neighborhoods that gives us roughly 50 train / 13 test.
# We set a random seed so the split is reproducible — anyone
# running this notebook gets the same split every time.
TRAIN_PCT    = 0.8
RANDOM_SEED  = 42

print("✓ Configuration set")
print(f"  Input:        {SUMMARY_PATH}")
print(f"  Output:       {FEATURE_TABLE_PATH}")
print(f"  Train/test:   {int(TRAIN_PCT*100)}/{int((1-TRAIN_PCT)*100)} split")
print(f"  Random seed:  {RANDOM_SEED}")

## Cell 3 — Load Summary Table

In [0]:
# Read the neighborhood summary serving table.
# This is the output of nb_Neighborhood_Summary.py — all metrics
# joined, Census values cleaned, one row per neighborhood.
# We use this as our starting point rather than re-reading raw
# data because the join and cleaning work is already done.
summary_df = spark.read.parquet(SUMMARY_PATH)

total = summary_df.count()
print(f"✓ Loaded summary table: {total} neighborhoods")
print(f"\nSchema:")
summary_df.printSchema()

## Cell 4 — Select Features and Target
This cell defines exactly which columns are features and which is the target variable. Keeping this explicit and separate from the transformation logic. 

**if someone wants to experiment with a different target variable or drop a feature, they only need to change this cell.**

In [0]:
# ── Target variable ───────────────────────────────────────────
# What we're trying to predict — average assessed property value
# per neighborhood. This is a continuous variable so this is a
# regression problem, not classification.
TARGET = "avg_assessed_value"

# ── Feature columns ───────────────────────────────────────────
# These are the inputs to the model. Each one captures a different
# dimension of neighborhood character:
#
# Crime features — safety signal
#   avg_monthly_incidents: overall crime volume
#   avg_violent_pct:       share of crime that is violent
#
# Parcel features — physical condition signal
#   vacancy_rate:          share of parcels sitting vacant
#
# Census features — socioeconomic signal
#   median_income:         area-weighted median household income
#   housing_cost_burden:   share of households spending >30% on housing
#
# Environmental features — risk signal
#   flood_zone_pct:        share of neighborhood in FEMA flood zone
#
# Weather features — climate context
#   avg_high_temp_f:       average daily high temperature
#   avg_daily_precip_mm:   average daily precipitation
#
# Note: we exclude avg_appraised_value and total_assessed_value
# because they are directly derived from avg_assessed_value and
# would cause data leakage — the model would essentially be
# predicting the target from itself.
FEATURE_COLS = [
    # Crime
    "avg_monthly_incidents",
    "avg_violent_pct",
    # Parcel
    "vacancy_rate",
    # Census
    "median_income",
    "housing_cost_burden",
    # Environmental
    "flood_zone_pct",
    # Weather
    "avg_high_temp_f",
    "avg_daily_precip_mm",
]

# Select only the columns we need — neighborhood ID, target, features
feature_df = summary_df.select(
    "neighborhood",
    TARGET,
    *FEATURE_COLS
)

# Drop any rows where target or any feature is null —
# ML models can't handle null values and imputation is
# out of scope for this pipeline stage
feature_df = feature_df.dropna()

print(f"✓ Features selected")
print(f"  Target:    {TARGET}")
print(f"  Features:  {len(FEATURE_COLS)}")
print(f"  Rows after dropping nulls: {feature_df.count()}")
print(f"\nFeature preview:")
display(feature_df.limit(5))

## Cell 5 — Scale Features to 0-1 Range
This is the most important ML preprocessing step. Raw features are on wildly different scales:  `median_income` is in the tens of thousands while `avg_violent_pct` is between 0 and 1. Without scaling, a model would give far more weight to income simply because its numbers are larger, not because it's more predictive. Min-max scaling puts everything on the same 0-1 scale so the model can make fair comparisons.

We keep both the raw and scaled columns in the output, raw for interpretability, scaled for model training.


In [0]:
# ── Compute min/max for each feature ─────────────────────────
# We need these across the full dataset to scale correctly.
# Collecting as a single row so we can use values as literals.
min_max_exprs = []
for col in FEATURE_COLS:
    min_max_exprs.append(F.min(col).alias(f"min_{col}"))
    min_max_exprs.append(F.max(col).alias(f"max_{col}"))

stats = feature_df.agg(*min_max_exprs).first()

# ── Apply min-max scaling ─────────────────────────────────────
# Formula: scaled = (value - min) / (max - min)
# Result is 0.0 for the neighborhood with the lowest value
# and 1.0 for the neighborhood with the highest value.
# All others fall linearly between them.
#
# We name scaled columns with a "_scaled" suffix so raw and
# scaled versions can coexist in the same table — downstream
# consumers can use whichever version they need.
scaled_df = feature_df
for col in FEATURE_COLS:
    min_val = float(stats[f"min_{col}"])
    max_val = float(stats[f"max_{col}"])

    if max_val == min_val:
        # All neighborhoods identical on this feature —
        # set scaled value to 0.5 (neutral) to avoid divide by zero
        scaled_df = scaled_df.withColumn(
            f"{col}_scaled",
            F.lit(0.5).cast(DoubleType())
        )
    else:
        scaled_df = scaled_df.withColumn(
            f"{col}_scaled",
            F.round(
                (F.col(col).cast(DoubleType()) - F.lit(min_val)) /
                F.lit(max_val - min_val),
                6   # 6 decimal places for ML precision
            )
        )

print(f"✓ Features scaled to 0-1 range")
print(f"\nScaled feature ranges (should all be 0.0 to 1.0):")
display(
    scaled_df.agg(*[
        F.round(F.min(f"{col}_scaled"), 4).alias(f"min_{col}")
        for col in FEATURE_COLS
    ] + [
        F.round(F.max(f"{col}_scaled"), 4).alias(f"max_{col}")
        for col in FEATURE_COLS
    ])
)

## Cell 6 — Train/Test Split
This cell assigns each neighborhood to either the training set or the test set. With only 63 neighborhoods we can't do a random row-level split the way you would with thousands of records, instead we use a deterministic hash of the neighborhood name to assign splits. This ensures the same neighborhood always lands in the same split regardless of how many times you re-run the notebook, which is important for reproducibility.

In [0]:
# ── Assign train/test split ───────────────────────────────────
# With only 63 neighborhoods a random row split could produce
# very uneven results. We use a hash of the neighborhood name
# modulo 100 to get a stable, reproducible split.
#
# How it works:
#   hash(neighborhood_name) → large integer
#   integer mod 100         → 0-99
#   if result < 80          → "train" (80% of neighborhoods)
#   if result >= 80         → "test"  (20% of neighborhoods)
#
# The same neighborhood always gets the same hash value so the
# split is identical on every run — critical for ML reproducibility.
# Changing RANDOM_SEED in Cell 2 shifts which neighborhoods
# land in train vs test without changing the overall proportions.

split_df = scaled_df.withColumn(
    "split",
    F.when(
        # xxhash64 is a fast deterministic hash function available
        # in Spark. We XOR with the seed to make it seed-dependent.
        (F.abs(F.xxhash64(F.col("neighborhood"))) +
         F.lit(RANDOM_SEED)) % 100 < F.lit(int(TRAIN_PCT * 100)),
        F.lit("train")
    ).otherwise(F.lit("test"))
)

# ── Verify split distribution ─────────────────────────────────
train_count = split_df.filter(F.col("split") == "train").count()
test_count  = split_df.filter(F.col("split") == "test").count()
total       = split_df.count()

print(f"✓ Train/test split applied")
print(f"  Train: {train_count} neighborhoods ({train_count/total*100:.1f}%)")
print(f"  Test:  {test_count} neighborhoods ({test_count/total*100:.1f}%)")

# Show which neighborhoods landed in test so it's transparent
print(f"\nTest set neighborhoods:")
display(
    split_df
    .filter(F.col("split") == "test")
    .select("neighborhood", TARGET, "split")
    .orderBy("neighborhood")
)

## Cell 7 — Final Column Organization
This cell arranges all columns into a logical, well-documented order before writing. Good column organization matters for a feature table because multiple people may consume it, a clear structure makes it immediately obvious what each column is and whether to use the raw or scaled version.

In [0]:
# ── Build final column list in logical order ──────────────────
# Group columns into sections so the table is self-documenting:
#   1. Identifier    — neighborhood name
#   2. Split         — train/test assignment
#   3. Target        — what the model predicts
#   4. Raw features  — original values for interpretability
#   5. Scaled features — 0-1 normalized values for model input

# Raw feature columns — same order as FEATURE_COLS for consistency
raw_cols    = FEATURE_COLS

# Scaled feature columns — same order with _scaled suffix
scaled_cols = [f"{col}_scaled" for col in FEATURE_COLS]

# Assemble final DataFrame in documented column order
feature_table = split_df.select(
    # ── Identifier ────────────────────────────────────────────
    # Not a model input — used to trace predictions back to
    # specific neighborhoods for validation and reporting
    F.col("neighborhood"),

    # ── Split indicator ───────────────────────────────────────
    # "train" or "test" — filters for model training vs evaluation
    F.col("split"),

    # ── Target variable ───────────────────────────────────────
    # avg_assessed_value — what the model is trained to predict
    # Units: USD (e.g. 45000 = $45,000 average assessed value)
    F.col(TARGET),

    # ── Raw feature columns ───────────────────────────────────
    # Original values — useful for interpretation and reporting.
    # Use these to explain model results in human-readable terms.
    *[F.col(c) for c in raw_cols],

    # ── Scaled feature columns ────────────────────────────────
    # Min-max normalized to 0.0-1.0 range across all neighborhoods.
    # Use these as direct model inputs — all features on same scale.
    *[F.col(c) for c in scaled_cols],
)

print(f"✓ Feature table finalized")
print(f"  Neighborhoods: {feature_table.count()}")
print(f"  Total columns: {len(feature_table.columns)}")
print(f"\nColumn inventory:")
for i, col in enumerate(feature_table.columns, 1):
    col_type = dict(feature_table.dtypes)[col]
    print(f"  {i:2d}. {col:40s} {col_type}")

## Cell 8 — Write Feature Table

In [0]:
# Ensure the serving volume exists before writing
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.serving")

# Write the ML feature table to Parquet.
# No partitioning needed — with 63 rows this is a single small
# file. Partitioning only adds overhead at this scale.
feature_table.write.mode("overwrite").parquet(FEATURE_TABLE_PATH)

# Read back to verify the write succeeded
written = spark.read.parquet(FEATURE_TABLE_PATH)
written_count = written.count()

print(f"✓ Wrote {written_count} rows → {FEATURE_TABLE_PATH}")

# Verify train/test split survived the write
train_written = written.filter(F.col("split") == "train").count()
test_written  = written.filter(F.col("split") == "test").count()
print(f"\n  Train rows: {train_written}")
print(f"  Test rows:  {test_written}")

if written_count == feature_table.count():
    print(f"\n✓ Row count matches — write verified")
else:
    print(f"\n⚠ Row count mismatch — check the write")

## Cell 9 — Quality Checks
The final validation before this table is considered done. Three checks that confirm the feature table is ML-ready: no nulls, features all in the right range, and the target variable distribution looks reasonable.

In [0]:
result = spark.read.parquet(FEATURE_TABLE_PATH)

# ── Check 1: Null rates ───────────────────────────────────────
# A feature table with nulls is not ML-ready. Any null here
# means something went wrong in the scaling or join steps.
# Expected: 0% nulls across all columns.
print("=== NULL RATES (%) ===")
display(result.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) /
        F.count(F.lit(1)) * 100, 2
    ).alias(c)
    for c in [
        "neighborhood", "split", TARGET,
        *FEATURE_COLS,
        *[f"{c}_scaled" for c in FEATURE_COLS]
    ]
]))

# COMMAND ----------

# ── Check 2: Scaled feature ranges ───────────────────────────
# Every scaled column should have a min of 0.0 and max of 1.0.
# If any column is outside this range the scaling failed.
print("=== SCALED FEATURE RANGES (min should be 0.0, max should be 1.0) ===")
display(
    result.agg(*[
        F.min(f"{col}_scaled").alias(f"min_{col}")
        for col in FEATURE_COLS
    ] + [
        F.max(f"{col}_scaled").alias(f"max_{col}")
        for col in FEATURE_COLS
    ])
)

# COMMAND ----------

# ── Check 3: Target variable distribution ────────────────────
# Shows the spread of avg_assessed_value across neighborhoods.
# A wide range is good — it means the model has meaningful
# variation to learn from. If all neighborhoods have similar
# values the regression problem becomes trivial.
print("=== TARGET VARIABLE DISTRIBUTION (avg_assessed_value) ===")
display(
    result.select(
        F.min(TARGET).alias("min"),
        F.round(F.avg(TARGET), 0).alias("avg"),
        F.max(TARGET).alias("max"),
        F.round(F.stddev(TARGET), 0).alias("std_dev"),
        F.percentile_approx(TARGET, 0.25).alias("p25"),
        F.percentile_approx(TARGET, 0.75).alias("p75"),
    )
)

# COMMAND ----------

# ── Check 4: Full feature table preview ──────────────────────
# Final review of the complete table — confirm it looks correct
# before presenting. Train rows first, then test.
print("=== FEATURE TABLE PREVIEW ===")
display(
    result
    .select(
        "neighborhood", "split", TARGET,
        "avg_monthly_incidents_scaled",
        "vacancy_rate_scaled",
        "median_income_scaled",
        "flood_zone_pct_scaled",
    )
    .orderBy("split", "neighborhood")
)